# AlphaLudo Google Colab Training

This notebook trains the AlphaLudo agent using Colab's GPU.

**Optimizations:**
- Uses CUDA for GPU acceleration
- Larger batch size (64)
- Increased MCTS simulations (400)
- **Custom Worker**: Optimized for Colab with progress tracking
- Resumes from local checkpoint (if included in zip)

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Install Dependencies
!pip install pybind11

In [ ]:
# 3. Upload Source Code
# Upload 'alphaludo_src.zip' here
from google.colab import files
import os

if not os.path.exists('alphaludo_src.zip'):
    print("Please upload alphaludo_src.zip")
    uploaded = files.upload()

In [ ]:
# 4. Setup Environment
!unzip -o alphaludo_src.zip
!pip install -e . --no-build-isolation

In [ ]:
# 5. Mount Google Drive (for Checkpoints)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 6. Training Configuration
import torch

# Optimizations for Colab / GPU
SP_BATCH_SIZE = 64      # Batch size for self-play
MCTS_SIMS = 400         # Simulations per move
BUFFER_SIZE = 100000    # Replay buffer size
ITERATIONS = 500        # Number of iterations to run
CHECKPOINT_DIR = '/content/drive/MyDrive/AlphaLudo_Checkpoints' # Save to Drive

if torch.cuda.is_available():
    DEVICE = 'cuda'
    print(f"Training on GPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = 'cpu'
    print("WARNING: Training on CPU. Enable GPU Runtime!")

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
# 7. Main Training Loop
import sys
sys.path.append('colab_package')

import numpy as np
from src.model_mastery import AlphaLudoTopNet
from src.trainer import Trainer
from src.vector_league_colab import VectorLeagueColabWorker  # USE OPTIMIZED WORKER
from src.replay_buffer_mastery import ReplayBufferMastery
from src.training_utils import EloTracker

# Initialize Components
model = AlphaLudoTopNet(num_res_blocks=10, num_channels=128).to(DEVICE)
trainer = Trainer(model, learning_rate=0.001, device=DEVICE)

# LOAD CHECKPOINT IF AVAILABLE
checkpoint_path = 'colab_package/model_latest.pt'
if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from zip: {checkpoint_path}")
    trainer.load_checkpoint(checkpoint_path)
    print(f"Resuming from Epoch {trainer.total_epochs}")
else:
    print("No checkpoint found in zip, starting fresh.")

replay_buffer = ReplayBufferMastery(max_size=BUFFER_SIZE)
elo_tracker = EloTracker()

ghost_pool = {} 

# Use Colab Worker
worker = VectorLeagueColabWorker(
    main_model=model,
    probabilities={'Main': 1.0}, # Start with self-play
    mcts_simulations=MCTS_SIMS,
    visualize=False, # Disable visualizer
    elo_tracker=elo_tracker,
    ghost_pool=ghost_pool
)

print("Starting Training Loop...")

for i in range(1, ITERATIONS + 1):
    print(f"\n=== Iteration {i}/{ITERATIONS} ===")
    
    # Self-Play
    examples, sp_results = worker.play_batch(batch_size=SP_BATCH_SIZE)
    replay_buffer.add(examples)
    print(f"Collected {len(examples)} examples. Buffer: {len(replay_buffer)}")
    
    # Training
    loss, p_loss, v_loss = trainer.train_epoch(replay_buffer, batch_size=256, num_batches=100)
    print(f"Loss: {loss:.4f} (Policy: {p_loss:.4f}, Value: {v_loss:.4f})")
    
    # Save Checkpoint to Drive
    if i % 1 == 0: # Save every iteration
        save_path = os.path.join(CHECKPOINT_DIR, 'model_latest.pt')
        trainer.save_checkpoint(save_path)
        print(f"Saved checkpoint to {save_path}")